# 27b — Catching Negative Repository Recovery

## Objetivo

Recuperar, **sin inventar identidades**, la mejor evidencia de repositorio disponible para los sitios de Catching que ya poseen etiquetas Nielsen negativas.

Este notebook se inserta antes de `28_supervised_baseline.ipynb` porque el notebook 27 concluyó:

```text
PAUSE_BEFORE_28_RECOVER_NEGATIVE_X
```

La lógica es:

```text
catching_targets.csv
→ sitios con Y = 0
→ evidencia de repositorio ya generada en notebooks 12–13
→ mejor candidato por sitio
→ decidir qué repositorios son técnicamente recuperables
→ preparar lista para extracción de X
```

No se modifica ninguna etiqueta y no se acepta automáticamente una coincidencia ambigua.


## 1. Configuración

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_colwidth", 500)

PROJECT_DIR = Path("..").resolve()

REPORTS_DIR = PROJECT_DIR / "reports" / "tables"
PUBLIC_INTERIM_DIR = PROJECT_DIR / "data" / "public_v2" / "interim"

SEMISUP_REPORTS_DIR = (
    REPORTS_DIR
    / "semisupervised"
)

SEMISUP_INTERIM_DIR = (
    PROJECT_DIR
    / "data"
    / "semisupervised"
    / "interim"
)

SEMISUP_REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TARGETS = [
    "H1_violation",
    "H4_violation",
    "H5_violation",
    "H6_violation"
]

print("PROJECT_DIR:", PROJECT_DIR)


PROJECT_DIR: /Users/uriel/Documents/machine_learning


## 2. Cargar negativos priorizados

In [2]:
negative_path = (
    SEMISUP_REPORTS_DIR
    / "semisupervised_negative_site_recovery_priority.csv"
)

assert negative_path.exists()

negative_rows = pd.read_csv(
    negative_path
)

print(
    "Negative label rows:",
    negative_rows.shape
)

print(
    "Unique negative sites:",
    negative_rows[
        "site_or_instance"
    ].nunique()
)

display(
    negative_rows
)


Negative label rows: (16, 5)
Unique negative sites: 12


,source_file,identity_column,site_or_instance,target,observed_label
0,data/public_v2/interim/catching_targets.csv,site,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,H1_violation,0
1,data/public_v2/interim/catching_targets.csv,site,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,H4_violation,0
2,data/public_v2/interim/catching_targets.csv,site,startbootstrap-freelancer,H4_violation,0
3,data/public_v2/interim/catching_targets.csv,site,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,H5_violation,0
4,data/public_v2/interim/catching_targets.csv,site,startbootstrap-resume,H5_violation,0
5,data/public_v2/interim/catching_targets.csv,site,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,H6_violation,0
6,data/public_v2/interim/catching_targets.csv,site,beginner-html-site-styled,H6_violation,0
7,data/public_v2/interim/catching_targets.csv,site,charity_responsive-template,H6_violation,0
8,data/public_v2/interim/catching_targets.csv,site,conference-website-template,H6_violation,0
9,data/public_v2/interim/catching_targets.csv,site,personal-website-template,H6_violation,0


## 3. Normalización de identidad

In [3]:
def normalize_identity(value):

    value = str(value).strip().lower()

    value = re.sub(
        r"[^a-z0-9]+",
        "",
        value
    )

    return value


negative_rows[
    "site_key"
] = negative_rows[
    "site_or_instance"
].map(
    normalize_identity
)

negative_sites = (
    negative_rows[
        [
            "site_or_instance",
            "site_key"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

display(
    negative_sites
)


,site_or_instance,site_key
0,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,electronicsecommerceshopwithadmindashboardnextjsnodejsmain
1,startbootstrap-freelancer,startbootstrapfreelancer
2,startbootstrap-resume,startbootstrapresume
3,beginner-html-site-styled,beginnerhtmlsitestyled
4,charity_responsive-template,charityresponsivetemplate
5,conference-website-template,conferencewebsitetemplate
6,personal-website-template,personalwebsitetemplate
7,react-saas-template,reactsaastemplate
8,simple-gallery_responsive-template,simplegalleryresponsivetemplate
9,single-page-small-business-web-application,singlepagesmallbusinesswebapplication


## 4. Cargar artefactos de recuperación de Catching

Se utilizan únicamente archivos producidos por las fases anteriores.

No se hace matching por posición de fila.


In [4]:
candidate_filenames = [
    "catching_repository_final.csv",
    "catching_source_recovery_review.csv",
    "catching_single_candidate_validation.csv",
    "catching_multiple_candidate_validation.csv",
    "catching_source_candidates.csv",
    "catching_sites_source_recovery.csv",
    "catching_source_recovery_status.csv",
    "catching_confirmed_repository_registry.csv",
    "catching_confirmed_repository_inventory.csv"
]

artifact_records = []

for filename in candidate_filenames:

    paths = []

    direct_report = (
        REPORTS_DIR
        /
        filename
    )

    direct_interim = (
        PUBLIC_INTERIM_DIR
        /
        filename
    )

    for path in [
        direct_report,
        direct_interim
    ]:
        if path.exists():
            paths.append(path)

    for root in [
        REPORTS_DIR,
        PUBLIC_INTERIM_DIR
    ]:
        if not root.exists():
            continue

        for path in root.rglob(
            filename
        ):
            if path.exists():
                paths.append(path)

    seen = set()

    for path in paths:

        key = str(
            path.resolve()
        )

        if key in seen:
            continue

        seen.add(key)

        try:
            df = pd.read_csv(
                path
            )
        except Exception:
            continue

        artifact_records.append({
            "filename":
                filename,

            "path":
                str(
                    path.relative_to(
                        PROJECT_DIR
                    )
                ),

            "rows":
                len(df),

            "columns":
                len(df.columns),

            "column_names":
                " | ".join(
                    df.columns.astype(str)
                )
        })

artifact_inventory = pd.DataFrame(
    artifact_records
)

display(
    artifact_inventory
)


,filename,path,rows,columns,column_names
0,catching_repository_final.csv,reports/tables/catching_repository_final.csv,30,3,site | repository_status | selected_repo
1,catching_source_recovery_review.csv,reports/tables/catching_source_recovery_review.csv,30,10,site | normalized_site | candidate_count | exact_match_count | recovery_status | confirmed_repo | evidence | exact_repositories | repositories | review_status
2,catching_single_candidate_validation.csv,reports/tables/catching_single_candidate_validation.csv,11,10,site | repo_full_name | clone_ok | source_files | snippets_tested | matched_snippets | matched_evaluations | matched_files | error | validation_status
3,catching_multiple_candidate_validation.csv,reports/tables/catching_multiple_candidate_validation.csv,62,10,site | repo_full_name | clone_ok | source_files | snippets_tested | matched_snippets | matched_evaluations | matched_files | error | validation_status
4,catching_source_candidates.csv,reports/tables/catching_source_candidates.csv,268,17,site | normalized_site | rank | repo_full_name | repo_name | repo_normalized | html_url | clone_url | default_branch | description | language | stars | fork | archived | license_key | license_name | exact_name_match
5,catching_sites_source_recovery.csv,reports/tables/catching_sites_source_recovery.csv,30,4,site | source_recovery_status | source_path | source_url
6,catching_source_recovery_status.csv,reports/tables/catching_source_recovery_status.csv,30,7,site | normalized_site | candidate_count | exact_match_count | recovery_status | confirmed_repo | evidence
7,catching_confirmed_repository_registry.csv,reports/tables/catching_confirmed_repository_registry.csv,2,16,site | repository_status | selected_repo | repo_full_name | clone_url | html_url | default_branch | license_key | license_name | local_path | clone_status | error | commit_sha | branch | commit_date | remote_url
8,catching_confirmed_repository_inventory.csv,reports/tables/catching_confirmed_repository_inventory.csv,69,5,site | repo_full_name | relative_path | extension | size_bytes


## Corrección aplicada antes del cierre

La primera ejecución reveló un problema en la detección automática de columnas:

`catching_repository_final.csv` contiene:

```text
site | repository_status | selected_repo
```

La función anterior podía seleccionar `repository_status` como si fuera la columna
del repositorio, produciendo valores falsos como:

```text
best_repository = PROVISIONAL_MODERATE
```

Se corrigió el detector para:

1. priorizar `selected_repo`, `confirmed_repo` y `repo_full_name`;
2. excluir columnas que contienen `status`, `decision`, `quality`, `evidence` o `count`;
3. mantener separadas la fuerza de evidencia y la identidad del repositorio.

Esta corrección puede cambiar el número y la identidad de los candidatos moderados.
Por ello todos los outputs anteriores fueron eliminados y el notebook debe ejecutarse
de nuevo antes de continuar.


## 5. Detectores de columnas

In [5]:
def detect_identity_column(df):

    preferred = [
        "site",
        "website",
        "source_site",
        "source_name",
        "template_name",
        "name"
    ]

    lower_map = {
        col.lower():
            col
        for col in df.columns
    }

    for candidate in preferred:

        if candidate in lower_map:
            return lower_map[
                candidate
            ]

    return None


def detect_status_column(df):

    preferred_exact = [
        "repository_status",
        "validation_status",
        "recovery_status",
        "source_recovery_status",
        "review_status",
        "match_quality",
        "decision"
    ]

    lower_map = {
        col.lower():
            col
        for col in df.columns
    }

    for candidate in preferred_exact:

        if candidate in lower_map:
            return lower_map[
                candidate
            ]

    candidates = [
        col
        for col in df.columns
        if (
            "status" in col.lower()
            or
            "decision" in col.lower()
            or
            "match_quality" in col.lower()
        )
    ]

    return (
        candidates[0]
        if candidates
        else None
    )


def detect_repo_column(df):

    # Priorizar columnas que contienen una identidad real de repositorio.
    # Excluir explícitamente columnas de estado/decisión para evitar
    # interpretar valores como PROVISIONAL_MODERATE como nombres de repo.
    preferred_exact = [
        "selected_repo",
        "confirmed_repo",
        "repo_full_name",
        "repository_full_name",
        "candidate_repo",
        "candidate_repository",
        "full_name",
        "repo",
        "repository"
    ]

    lower_map = {
        col.lower():
            col
        for col in df.columns
    }

    for candidate in preferred_exact:

        if candidate in lower_map:
            return lower_map[
                candidate
            ]

    excluded_tokens = [
        "status",
        "decision",
        "quality",
        "evidence",
        "count"
    ]

    repo_candidates = [
        col
        for col in df.columns
        if (
            "repo" in col.lower()
            and
            "url" not in col.lower()
            and
            not any(
                token in col.lower()
                for token in excluded_tokens
            )
        )
    ]

    return (
        repo_candidates[0]
        if repo_candidates
        else None
    )


def detect_url_column(df):

    candidates = [
        col
        for col in df.columns
        if (
            "url" in col.lower()
            and
            (
                "repo" in col.lower()
                or
                "github" in col.lower()
                or
                col.lower() == "url"
            )
        )
    ]

    return (
        candidates[0]
        if candidates
        else None
    )


## 6. Consolidar evidencia para sitios negativos

In [6]:
evidence_records = []

for _, artifact in (
    artifact_inventory.iterrows()
):

    path = (
        PROJECT_DIR
        /
        artifact[
            "path"
        ]
    )

    try:
        df = pd.read_csv(
            path
        )
    except Exception:
        continue

    identity_col = detect_identity_column(
        df
    )

    if identity_col is None:
        continue

    status_col = detect_status_column(
        df
    )

    repo_col = detect_repo_column(
        df
    )

    url_col = detect_url_column(
        df
    )

    working = df.copy()

    working[
        "__site_key"
    ] = working[
        identity_col
    ].map(
        normalize_identity
    )

    working = working[
        working[
            "__site_key"
        ].isin(
            set(
                negative_sites[
                    "site_key"
                ]
            )
        )
    ]

    for _, row in (
        working.iterrows()
    ):

        evidence_records.append({
            "site_key":
                row[
                    "__site_key"
                ],

            "site_value_in_artifact":
                str(
                    row[
                        identity_col
                    ]
                ),

            "artifact_path":
                artifact[
                    "path"
                ],

            "status":
                (
                    str(
                        row[
                            status_col
                        ]
                    )
                    if status_col is not None
                    else ""
                ),

            "repository":
                (
                    str(
                        row[
                            repo_col
                        ]
                    )
                    if repo_col is not None
                    else ""
                ),

            "repository_url":
                (
                    str(
                        row[
                            url_col
                        ]
                    )
                    if url_col is not None
                    else ""
                ),

            "identity_column":
                identity_col,

            "status_column":
                (
                    status_col
                    if status_col is not None
                    else ""
                ),

            "repo_column":
                (
                    repo_col
                    if repo_col is not None
                    else ""
                ),

            "url_column":
                (
                    url_col
                    if url_col is not None
                    else ""
                )
        })

evidence = pd.DataFrame(
    evidence_records
)

print(
    "Evidence rows:",
    len(
        evidence
    )
)

display(
    evidence
)


Evidence rows: 210


,site_key,site_value_in_artifact,artifact_path,status,repository,repository_url,identity_column,status_column,repo_column,url_column
0,electronicsecommerceshopwithadmindashboardnextjsnodejsmain,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,reports/tables/catching_repository_final.csv,UNRESOLVED,nan,,site,repository_status,selected_repo,
1,beginnerhtmlsitestyled,beginner-html-site-styled,reports/tables/catching_repository_final.csv,AMBIGUOUS_MULTIPLE_MODERATE,nan,,site,repository_status,selected_repo,
2,charityresponsivetemplate,charity_responsive-template,reports/tables/catching_repository_final.csv,UNRESOLVED,nan,,site,repository_status,selected_repo,
3,conferencewebsitetemplate,conference-website-template,reports/tables/catching_repository_final.csv,PROVISIONAL_MODERATE,mikepierce/conference-website-template,,site,repository_status,selected_repo,
4,personalwebsitetemplate,personal-website-template,reports/tables/catching_repository_final.csv,PROVISIONAL_MODERATE,The-WebOps-Club/personal-website-template,,site,repository_status,selected_repo,
5,reactsaastemplate,react-saas-template,reports/tables/catching_repository_final.csv,PROVISIONAL_MODERATE,dunky11/react-saas-template,,site,repository_status,selected_repo,
6,simplegalleryresponsivetemplate,simple-gallery_responsive-template,reports/tables/catching_repository_final.csv,UNRESOLVED,nan,,site,repository_status,selected_repo,
7,singlepagesmallbusinesswebapplication,single-page-small-business-web-application,reports/tables/catching_repository_final.csv,UNRESOLVED,nan,,site,repository_status,selected_repo,
8,startbootstrapfreelancer,startbootstrap-freelancer,reports/tables/catching_repository_final.csv,UNRESOLVED,nan,,site,repository_status,selected_repo,
9,startbootstraplandingpage,startbootstrap-landing-page,reports/tables/catching_repository_final.csv,UNRESOLVED,nan,,site,repository_status,selected_repo,


## 7. Normalizar estados y asignar fuerza de evidencia

La jerarquía es conservadora:

```text
CONFIRMED_STRONG
> PROVISIONAL_MODERATE / MODERATE_MATCH
> AMBIGUOUS_MULTIPLE_MODERATE
> UNKNOWN
> NO_MATCH / UNRESOLVED / NO_TESTABLE_EVIDENCE
```

Solo `CONFIRMED_STRONG` se considera automáticamente apto para join científico.

Los estados moderados quedan como candidatos a validación técnica adicional.


In [7]:
def classify_status(value):

    text = str(
        value
    ).strip().upper()

    if (
        "CONFIRMED_STRONG"
        in text
        or
        text == "STRONG"
    ):
        return (
            "CONFIRMED_STRONG",
            5
        )

    if (
        "PROVISIONAL_MODERATE"
        in text
        or
        "MODERATE_MATCH"
        in text
    ):
        return (
            "MODERATE",
            4
        )

    if (
        "AMBIGUOUS"
        in text
        and
        "MODERATE"
        in text
    ):
        return (
            "AMBIGUOUS_MODERATE",
            3
        )

    if (
        text == ""
        or
        text == "NAN"
        or
        "UNKNOWN"
        in text
    ):
        return (
            "UNKNOWN",
            2
        )

    if (
        "NO_MATCH"
        in text
        or
        "UNRESOLVED"
        in text
        or
        "NO_TESTABLE_EVIDENCE"
        in text
    ):
        return (
            "NO_EVIDENCE",
            1
        )

    return (
        "OTHER",
        2
    )


if evidence.empty:

    evidence[
        "normalized_status"
    ] = []

    evidence[
        "status_rank"
    ] = []

else:

    parsed = evidence[
        "status"
    ].map(
        classify_status
    )

    evidence[
        "normalized_status"
    ] = [
        value[0]
        for value in parsed
    ]

    evidence[
        "status_rank"
    ] = [
        value[1]
        for value in parsed
    ]

display(
    evidence[
        [
            "site_key",
            "artifact_path",
            "status",
            "normalized_status",
            "status_rank",
            "repository",
            "repository_url"
        ]
    ]
)


,site_key,artifact_path,status,normalized_status,status_rank,repository,repository_url
0,electronicsecommerceshopwithadmindashboardnextjsnodejsmain,reports/tables/catching_repository_final.csv,UNRESOLVED,NO_EVIDENCE,1,nan,
1,beginnerhtmlsitestyled,reports/tables/catching_repository_final.csv,AMBIGUOUS_MULTIPLE_MODERATE,AMBIGUOUS_MODERATE,3,nan,
2,charityresponsivetemplate,reports/tables/catching_repository_final.csv,UNRESOLVED,NO_EVIDENCE,1,nan,
3,conferencewebsitetemplate,reports/tables/catching_repository_final.csv,PROVISIONAL_MODERATE,MODERATE,4,mikepierce/conference-website-template,
4,personalwebsitetemplate,reports/tables/catching_repository_final.csv,PROVISIONAL_MODERATE,MODERATE,4,The-WebOps-Club/personal-website-template,
5,reactsaastemplate,reports/tables/catching_repository_final.csv,PROVISIONAL_MODERATE,MODERATE,4,dunky11/react-saas-template,
6,simplegalleryresponsivetemplate,reports/tables/catching_repository_final.csv,UNRESOLVED,NO_EVIDENCE,1,nan,
7,singlepagesmallbusinesswebapplication,reports/tables/catching_repository_final.csv,UNRESOLVED,NO_EVIDENCE,1,nan,
8,startbootstrapfreelancer,reports/tables/catching_repository_final.csv,UNRESOLVED,NO_EVIDENCE,1,nan,
9,startbootstraplandingpage,reports/tables/catching_repository_final.csv,UNRESOLVED,NO_EVIDENCE,1,nan,


## 8. Resolver mejor evidencia por sitio

In [8]:
best_records = []

for _, site_row in (
    negative_sites.iterrows()
):

    site_key = site_row[
        "site_key"
    ]

    site_name = site_row[
        "site_or_instance"
    ]

    subset = evidence[
        evidence[
            "site_key"
        ]
        ==
        site_key
    ].copy()

    if subset.empty:

        best_records.append({
            "site_or_instance":
                site_name,

            "site_key":
                site_key,

            "best_status":
                "NO_EVIDENCE",

            "best_status_rank":
                0,

            "best_repository":
                "",

            "best_repository_url":
                "",

            "best_artifact_path":
                "",

            "scientific_join_ready":
                False,

            "requires_additional_validation":
                True
        })

        continue

    subset[
        "has_repository"
    ] = (
        subset[
            "repository"
        ]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
        &
        subset[
            "repository"
        ]
        .fillna("")
        .astype(str)
        .str.lower()
        .ne("nan")
    )

    subset = subset.sort_values(
        [
            "status_rank",
            "has_repository"
        ],
        ascending=[
            False,
            False
        ]
    )

    best = subset.iloc[0]

    scientific_join_ready = (
        best[
            "normalized_status"
        ]
        ==
        "CONFIRMED_STRONG"
        and
        bool(
            best[
                "has_repository"
            ]
        )
    )

    best_records.append({
        "site_or_instance":
            site_name,

        "site_key":
            site_key,

        "best_status":
            best[
                "normalized_status"
            ],

        "best_status_rank":
            int(
                best[
                    "status_rank"
                ]
            ),

        "best_repository":
            best[
                "repository"
            ],

        "best_repository_url":
            best[
                "repository_url"
            ],

        "best_artifact_path":
            best[
                "artifact_path"
            ],

        "scientific_join_ready":
            scientific_join_ready,

        "requires_additional_validation":
            not scientific_join_ready
    })

best_repository_evidence = pd.DataFrame(
    best_records
)

display(
    best_repository_evidence
)


,site_or_instance,site_key,best_status,best_status_rank,best_repository,best_repository_url,best_artifact_path,scientific_join_ready,requires_additional_validation
0,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,electronicsecommerceshopwithadmindashboardnextjsnodejsmain,UNKNOWN,2,Rajsu12/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,,reports/tables/catching_source_candidates.csv,False,True
1,startbootstrap-freelancer,startbootstrapfreelancer,UNKNOWN,2,StartBootstrap/startbootstrap-freelancer,,reports/tables/catching_source_candidates.csv,False,True
2,startbootstrap-resume,startbootstrapresume,UNKNOWN,2,StartBootstrap/startbootstrap-resume,,reports/tables/catching_source_candidates.csv,False,True
3,beginner-html-site-styled,beginnerhtmlsitestyled,MODERATE,4,mdn/beginner-html-site-styled,,reports/tables/catching_multiple_candidate_validation.csv,False,True
4,charity_responsive-template,charityresponsivetemplate,UNKNOWN,2,website-templates/charity_responsive-template,,reports/tables/catching_source_candidates.csv,False,True
5,conference-website-template,conferencewebsitetemplate,MODERATE,4,mikepierce/conference-website-template,,reports/tables/catching_repository_final.csv,False,True
6,personal-website-template,personalwebsitetemplate,MODERATE,4,The-WebOps-Club/personal-website-template,,reports/tables/catching_repository_final.csv,False,True
7,react-saas-template,reactsaastemplate,MODERATE,4,dunky11/react-saas-template,,reports/tables/catching_repository_final.csv,False,True
8,simple-gallery_responsive-template,simplegalleryresponsivetemplate,UNKNOWN,2,website-templates/simple-gallery_responsive-template,,reports/tables/catching_source_candidates.csv,False,True
9,single-page-small-business-web-application,singlepagesmallbusinesswebapplication,UNKNOWN,2,reza-azimifar/single-page-small-business-web-application,,reports/tables/catching_source_candidates.csv,False,True


## 9. Adjuntar targets negativos por sitio

In [9]:
negative_target_summary = (
    negative_rows
    .groupby(
        [
            "site_or_instance",
            "site_key"
        ]
    )[
        "target"
    ]
    .apply(
        lambda x:
            " | ".join(
                sorted(
                    set(
                        x.astype(str)
                    )
                )
            )
    )
    .reset_index(
        name="negative_targets"
    )
)

recovery_registry = (
    best_repository_evidence
    .merge(
        negative_target_summary,
        on=[
            "site_or_instance",
            "site_key"
        ],
        how="left",
        validate="one_to_one"
    )
)

display(
    recovery_registry
)


,site_or_instance,site_key,best_status,best_status_rank,best_repository,best_repository_url,best_artifact_path,scientific_join_ready,requires_additional_validation,negative_targets
0,Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,electronicsecommerceshopwithadmindashboardnextjsnodejsmain,UNKNOWN,2,Rajsu12/Electronics-eCommerce-Shop-With-Admin-Dashboard-NextJS-NodeJS-main,,reports/tables/catching_source_candidates.csv,False,True,H1_violation | H4_violation | H5_violation | H6_violation
1,startbootstrap-freelancer,startbootstrapfreelancer,UNKNOWN,2,StartBootstrap/startbootstrap-freelancer,,reports/tables/catching_source_candidates.csv,False,True,H4_violation | H6_violation
2,startbootstrap-resume,startbootstrapresume,UNKNOWN,2,StartBootstrap/startbootstrap-resume,,reports/tables/catching_source_candidates.csv,False,True,H5_violation
3,beginner-html-site-styled,beginnerhtmlsitestyled,MODERATE,4,mdn/beginner-html-site-styled,,reports/tables/catching_multiple_candidate_validation.csv,False,True,H6_violation
4,charity_responsive-template,charityresponsivetemplate,UNKNOWN,2,website-templates/charity_responsive-template,,reports/tables/catching_source_candidates.csv,False,True,H6_violation
5,conference-website-template,conferencewebsitetemplate,MODERATE,4,mikepierce/conference-website-template,,reports/tables/catching_repository_final.csv,False,True,H6_violation
6,personal-website-template,personalwebsitetemplate,MODERATE,4,The-WebOps-Club/personal-website-template,,reports/tables/catching_repository_final.csv,False,True,H6_violation
7,react-saas-template,reactsaastemplate,MODERATE,4,dunky11/react-saas-template,,reports/tables/catching_repository_final.csv,False,True,H6_violation
8,simple-gallery_responsive-template,simplegalleryresponsivetemplate,UNKNOWN,2,website-templates/simple-gallery_responsive-template,,reports/tables/catching_source_candidates.csv,False,True,H6_violation
9,single-page-small-business-web-application,singlepagesmallbusinesswebapplication,UNKNOWN,2,reza-azimifar/single-page-small-business-web-application,,reports/tables/catching_source_candidates.csv,False,True,H6_violation


## 10. Detectar repositorios moderados con nombre utilizable

Una coincidencia `MODERATE` **no se convierte automáticamente en fuerte**.

Se separan para la siguiente fase técnica, donde deberá comprobarse:

- repositorio accesible;
- contenido web real;
- correspondencia del template/proyecto;
- capacidad de extraer las mismas X.

No se usará un candidato moderado en entrenamiento sin dejar su procedencia explícita.


In [10]:
moderate_candidates = recovery_registry[
    recovery_registry[
        "best_status"
    ]
    ==
    "MODERATE"
].copy()

moderate_candidates[
    "repository_name_available"
] = (
    moderate_candidates[
        "best_repository"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
    &
    moderate_candidates[
        "best_repository"
    ]
    .fillna("")
    .astype(str)
    .str.lower()
    .ne("nan")
)

display(
    moderate_candidates
)


,site_or_instance,site_key,best_status,best_status_rank,best_repository,best_repository_url,best_artifact_path,scientific_join_ready,requires_additional_validation,negative_targets,repository_name_available
3,beginner-html-site-styled,beginnerhtmlsitestyled,MODERATE,4,mdn/beginner-html-site-styled,,reports/tables/catching_multiple_candidate_validation.csv,False,True,H6_violation,True
5,conference-website-template,conferencewebsitetemplate,MODERATE,4,mikepierce/conference-website-template,,reports/tables/catching_repository_final.csv,False,True,H6_violation,True
6,personal-website-template,personalwebsitetemplate,MODERATE,4,The-WebOps-Club/personal-website-template,,reports/tables/catching_repository_final.csv,False,True,H6_violation,True
7,react-saas-template,reactsaastemplate,MODERATE,4,dunky11/react-saas-template,,reports/tables/catching_repository_final.csv,False,True,H6_violation,True


## 11. Resumen por target

In [11]:
target_recovery_records = []

for target in TARGETS:

    sites_for_target = set(
        negative_rows.loc[
            negative_rows[
                "target"
            ]
            ==
            target,
            "site_or_instance"
        ]
        .astype(str)
        .tolist()
    )

    subset = recovery_registry[
        recovery_registry[
            "site_or_instance"
        ].isin(
            sites_for_target
        )
    ]

    target_recovery_records.append({
        "target":
            target,

        "negative_sites":
            len(
                sites_for_target
            ),

        "confirmed_strong_sites":
            int(
                (
                    subset[
                        "best_status"
                    ]
                    ==
                    "CONFIRMED_STRONG"
                ).sum()
            ),

        "moderate_sites":
            int(
                (
                    subset[
                        "best_status"
                    ]
                    ==
                    "MODERATE"
                ).sum()
            ),

        "other_or_no_evidence_sites":
            int(
                (
                    ~subset[
                        "best_status"
                    ].isin(
                        [
                            "CONFIRMED_STRONG",
                            "MODERATE"
                        ]
                    )
                ).sum()
            ),

        "x_recovery_ready_strict":
            bool(
                (
                    subset[
                        "scientific_join_ready"
                    ]
                ).any()
            )
    })

target_recovery_summary = pd.DataFrame(
    target_recovery_records
)

display(
    target_recovery_summary
)


,target,negative_sites,confirmed_strong_sites,moderate_sites,other_or_no_evidence_sites,x_recovery_ready_strict
0,H1_violation,1,0,0,1,False
1,H4_violation,2,0,0,2,False
2,H5_violation,2,0,0,2,False
3,H6_violation,11,0,4,7,False


## 12. Decisión de cierre

Tres resultados posibles:

### `STRICT_NEGATIVE_REPOSITORY_READY`
Existe al menos un negativo con repositorio `CONFIRMED_STRONG`.

### `MODERATE_NEGATIVE_REPOSITORY_CANDIDATES`
No hay negativos fuertes, pero hay repositorios moderados técnicamente verificables.

### `NO_NEGATIVE_REPOSITORY_RECOVERY_PATH`
Los artefactos anteriores no ofrecen una ruta actual para recuperar X negativa.


In [12]:
strict_ready = bool(
    recovery_registry[
        "scientific_join_ready"
    ].any()
)

moderate_ready = bool(
    (
        moderate_candidates[
            "repository_name_available"
        ]
    ).any()
    if not moderate_candidates.empty
    else False
)

if strict_ready:

    global_status = (
        "STRICT_NEGATIVE_REPOSITORY_READY"
    )

    next_action = (
        "Extract the same structural X from "
        "confirmed negative repositories."
    )

elif moderate_ready:

    global_status = (
        "MODERATE_NEGATIVE_REPOSITORY_CANDIDATES"
    )

    next_action = (
        "Technically validate moderate repositories "
        "before extracting X."
    )

else:

    global_status = (
        "NO_NEGATIVE_REPOSITORY_RECOVERY_PATH"
    )

    next_action = (
        "Current repository evidence is insufficient "
        "for negative X recovery without new evidence."
    )

recovery_decision = pd.DataFrame([
    {
        "global_status":
            global_status,

        "strict_negative_repo_ready":
            strict_ready,

        "moderate_negative_candidates":
            moderate_ready,

        "next_action":
            next_action
    }
])

display(
    recovery_decision
)


,global_status,strict_negative_repo_ready,moderate_negative_candidates,next_action
0,MODERATE_NEGATIVE_REPOSITORY_CANDIDATES,False,True,Technically validate moderate repositories before extracting X.


## 13. Guardar resultados

In [13]:
artifact_inventory.to_csv(
    SEMISUP_REPORTS_DIR
    / "catching_negative_recovery_artifact_inventory.csv",
    index=False
)

evidence.to_csv(
    SEMISUP_REPORTS_DIR
    / "catching_negative_repository_evidence_long.csv",
    index=False
)

recovery_registry.to_csv(
    SEMISUP_REPORTS_DIR
    / "catching_negative_repository_recovery_registry.csv",
    index=False
)

moderate_candidates.to_csv(
    SEMISUP_REPORTS_DIR
    / "catching_negative_moderate_candidates.csv",
    index=False
)

target_recovery_summary.to_csv(
    SEMISUP_REPORTS_DIR
    / "catching_negative_recovery_by_target.csv",
    index=False
)

recovery_decision.to_csv(
    SEMISUP_REPORTS_DIR
    / "catching_negative_repository_recovery_decision.csv",
    index=False
)


In [14]:
expected_outputs = [
    "catching_negative_recovery_artifact_inventory.csv",
    "catching_negative_repository_evidence_long.csv",
    "catching_negative_repository_recovery_registry.csv",
    "catching_negative_moderate_candidates.csv",
    "catching_negative_recovery_by_target.csv",
    "catching_negative_repository_recovery_decision.csv"
]

for filename in expected_outputs:

    path = (
        SEMISUP_REPORTS_DIR
        /
        filename
    )

    print(
        filename,
        "✅"
        if path.exists()
        else "❌"
    )


catching_negative_recovery_artifact_inventory.csv ✅
catching_negative_repository_evidence_long.csv ✅
catching_negative_repository_recovery_registry.csv ✅
catching_negative_moderate_candidates.csv ✅
catching_negative_recovery_by_target.csv ✅
catching_negative_repository_recovery_decision.csv ✅


## Punto de detención

No pasar todavía a `28_supervised_baseline.ipynb`.

Si el resultado es:

```text
STRICT_NEGATIVE_REPOSITORY_READY
```

el siguiente notebook extraerá las tres X estructurales de esos repositorios.

Si el resultado es:

```text
MODERATE_NEGATIVE_REPOSITORY_CANDIDATES
```

se hará primero una validación técnica dirigida de esos candidatos.

Si el resultado es:

```text
NO_NEGATIVE_REPOSITORY_RECOVERY_PATH
```

habrá que reconsiderar la evidencia disponible antes de entrenar; no se crearán negativos sintéticos para resolver el problema.
